In [1]:
# %load_ext cudf.pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import time

from tqdm import tqdm
from copy import deepcopy
from glob import glob

In [2]:
notes_dataset = pd.read_csv('../Data/MBS_clinical_notes.csv')

In [4]:
notes_dataset.columns

Index(['mrn', 'clinical_note_id', 'clinical_note_instance_id',
       'clinically_significant_update_date_time',
       'clinically_significant_update_date_time_utc', 'valid_from_date_time',
       'valid_from_date_time_utc', 'order_id', 'accession_number', 'catalog',
       'encounter_id', 'event_start_date_time', 'event_end_date_time',
       'view_level', 'normalcy', 'valid_until_date_time',
       'valid_until_date_time_utc', 'note_result_status',
       'performed_by_personnel_id', 'event', 'event_class', 'event_date_time',
       'note_flag', 'note_status', 'note_text', 'note_pointer',
       'note_loaded_date_time_utc', 'note_refreshed_date_time_utc',
       'note_file_extension'],
      dtype='object')

In [7]:
notes_dataset.head()

,mrn,clinical_note_id,clinical_note_instance_id,clinically_significant_update_date_time,clinically_significant_update_date_time_utc,valid_from_date_time,valid_from_date_time_utc,order_id,accession_number,catalog,...,event,event_class,event_date_time,note_flag,note_status,note_text,note_pointer,note_loaded_date_time_utc,note_refreshed_date_time_utc,note_file_extension
0,ea99d6473bc63684e7bbe124fde827f00a8c872664b2e8...,7166279555,7166278882,2025-10-17T09:54:16.000Z,2025-10-17T13:54:16.000Z,2025-10-17T09:54:16.000Z,2025-10-17T13:54:16.000Z,3784406359,NaN,,...,Report,DOC,2025-10-16T20:18:00.000Z,False,COMPLETE,NaN,part-00003-tid-7424903864412702486-2562c76a-48...,2025-10-21T18:13:11.167Z,2025-10-21T18:31:23.021Z,FAILED TO PARSE
1,385f45a7a4d2de68a002383c73d69fe9703a419ee008cd...,7018649623,7018649655,2025-07-10T13:57:08.000Z,2025-07-10T17:57:08.000Z,2025-07-10T13:57:08.000Z,2025-07-10T17:57:08.000Z,3683178487,NaN,,...,Report,DOC,2025-07-10T17:03:00.000Z,False,COMPLETE,"Children's National Hospital - DC, Maryland, V...",dbfs:/Volumes/devtest_edp_silver_curated/mille...,2025-10-24T20:16:30.467Z,2025-10-24T20:27:16.653Z,txt
2,a3d15fc57e065a63e7a4250a4df22f214ecae7b66016b4...,6967378657,6974411954,2025-06-10T00:02:17.000Z,2025-06-10T04:02:17.000Z,2025-06-10T00:02:17.000Z,2025-06-10T04:02:17.000Z,0,NaN,,...,ED Note-Provider,DOC,2025-06-05T00:51:00.000Z,False,COMPLETE,"Patient: GRIFFITH, LOVE MRN: 1012...",/Volumes/devtest_edp_silver_curated/millennium...,2025-10-16T17:24:58.388Z,2025-10-16T17:49:05.249Z,rtf
3,16305a7fc891c515acc31f414998411d498bb613234c6e...,7155974792,7155972769,2025-10-10T11:30:16.000Z,2025-10-10T15:30:16.000Z,2025-10-10T11:30:16.000Z,2025-10-10T15:30:16.000Z,3781453377,NaN,,...,Report,DOC,2025-10-10T12:17:00.000Z,False,COMPLETE,"Children's National Hospital - DC, Maryland, V...",dbfs:/Volumes/devtest_edp_silver_curated/mille...,2025-10-24T19:37:19.944Z,2025-10-24T19:45:46.846Z,txt
4,5187d5e0eb33147269bc6cce35c6865b4afcbb06d8ace6...,7124788297,7125028212,2025-09-20T15:55:11.000Z,2025-09-20T19:55:11.000Z,2025-09-20T15:55:11.000Z,2025-09-20T19:55:11.000Z,0,NaN,,...,ED Note-Provider,DOC,2025-09-20T13:43:00.000Z,False,COMPLETE,"Patient: DEVOLDER, DANTE MRN: 101...",/Volumes/devtest_edp_silver_curated/millennium...,2025-10-16T17:11:12.933Z,2025-10-16T17:18:15.870Z,rtf


In [10]:
notes_dataset['event'].unique()

array(['Report', 'ED Note-Provider', 'Progress Notes',
       'Provider Letter - Distributed'], dtype=object)

In [5]:
notes_dataset_subset = notes_dataset[notes_dataset['mrn']==notes_dataset['mrn'][0]]

In [5]:
notes_dataset_subset['mrn'].unique()

array(['ea99d6473bc63684e7bbe124fde827f00a8c872664b2e83c7554c2b8163773a2'],
      dtype=object)

In [6]:
len(notes_dataset_subset)

6

In [7]:
for idx, row in notes_dataset_subset.iterrows():

    print("\n\n############ Note:", idx, " ############")
    print(row['note_text'])



############ Note: 0  ############
nan


############ Note: 42801  ############
Children's National Hospital - DC, Maryland, Virginia

XR Forearm RT

Pt. Name: OWENS, EXEL     Sex: M        D.O.B: 03/27/2020
MRN: 021597046        Accession #: XR20250092424
Completion Date: 09/18/2025 16:57
Ordered by: MADELINE PRICHARD

CLINICAL HISTORY: pain/deformity/injury

TECHNIQUE: AP and lateral radiographs

COMPARISON: Right forearm radiographs August 31, 2025

FINDINGS:
Bones: Soft tissue cast limits evaluation of fine details. Redemonstrated
fractures of the proximal radial and ulnar diaphyses with similar near-anatomic
alignment and residual mild volar apex angulation of the radius. Interval
healing, as evidenced by bridging callus formation.
Joint alignment: Normal
Joint fluid: Not well evaluated due to overlying soft tissue cast.
Soft tissues: Soft tissue cast in place.

IMPRESSION:

Healing proximal diaphyseal radial and ulnar fractures.

END OF IMPRESSION

Contributing Provider: Hwa-Py

In [8]:
notes = notes_dataset['note_text']

In [8]:
print(notes[150])

Children's National Hospital - DC, Maryland, Virginia

DXA Bone Density Axial

Pt. Name: SOLOMON, RACHEL     Sex: F        D.O.B: 10/05/2010
MRN: 021181102        Accession #: XR20250088476
Completion Date: 09/09/2025 09:55
Ordered by: LAURA TOSI

INDICATION:  lbd 

COMPARISON:  None

TECHNIQUE: Bone mineral densitometry examination has been performed on the
Horizon A (S/N201242) densitometer, software version 13.6.0.4
 
FINDINGS:
Patient height: 165 cm
Patient weight: 53 kg
Fat content: 33 % of total body weight

DXA Results Summary:
VFA: Not performed. .

Lumbar spine L1-4  
BMD 0.858 (g/cm2)
Z-score -0.9
90 AM (%)
 
Left total hip
BMD 0.785 (g/cm2)
Z-score -1.4
84 AM (%)
 
Total body less head:
BMD 0.783 (g/cm2)
Z-score -2.1
84 AM (%)

Left lateral distal femur
BMD 0.929 (g/cm2)
Z-score ranges from -1.3 through -1.2

Right lateral distal femur
BMD 0.958 (g/cm2)
Z-score ranges from -1.7 through -0.7
 
Impression :  
1. Low bone mineral density.
 
* According to the pediatric position

In [14]:
notes_dataset.columns

Index(['mrn', 'clinical_note_id', 'clinical_note_instance_id',
       'clinically_significant_update_date_time',
       'clinically_significant_update_date_time_utc', 'valid_from_date_time',
       'valid_from_date_time_utc', 'order_id', 'accession_number', 'catalog',
       'encounter_id', 'event_start_date_time', 'event_end_date_time',
       'view_level', 'normalcy', 'valid_until_date_time',
       'valid_until_date_time_utc', 'note_result_status',
       'performed_by_personnel_id', 'event', 'event_class', 'event_date_time',
       'note_flag', 'note_status', 'note_text', 'note_pointer',
       'note_loaded_date_time_utc', 'note_refreshed_date_time_utc',
       'note_file_extension'],
      dtype='object')

In [15]:
notes_dataset['note_pointer'][1]

'dbfs:/Volumes/devtest_edp_silver_curated/millennium_notes/raw_notes/clinical_note_id=7018649623/part-00007-tid-6449916956367060539-bbe6e23e-7802-4451-a222-850e38d20c03-4092-383.c000.txt'

In [13]:
notes_dataset["mrn"].nunique()

33416

In [ ]:
olivia rayford